In [1]:
!pip install mysql-connector-python faker --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\USUARIO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ✅ Importar librerias
import mysql.connector
from faker import Faker
import random
from datetime import date, timedelta, datetime

fake = Faker('es_CO') # Locale en español colombiano
random.seed(42)
Faker.seed(42)

In [4]:
# ✅ CONFIGURACIÓN DE CONEXIÓN A MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="centro_estetico" # nombre de tu base de datos
)
cursor = conn.cursor()
print("✅ Conexión exitosa")

✅ Conexión exitosa


In [6]:
# ── 3. FUNCIONES GENERADORAS DE IDs PERSONALIZADOS ──────────
# Estas funciones construyen el ID con prefijo + número con ceros
# a la izquierda (zfill), ejemplo: P0001, C0003, PS0002, I0010.
# Se consulta la tabla antes de insertar para calcular
# el siguiente número disponible de forma dinámica.

def siguiente_id(tabla, columna_pk, prefijo, digitos=4):
    """
    Genera el próximo ID con prefijo para una tabla dada.
    - tabla       : nombre de la tabla SQL
    - columna_pk  : nombre de la columna PK
    - prefijo     : letra(s) iniciales (ej: 'P', 'C', 'PS', 'I')
    - digitos     : cantidad de dígitos numéricos (default 4 → 0001)
    """
    cursor.execute(f"SELECT COUNT(*) FROM {tabla}")
    total = cursor.fetchone()[0]
    numero = total + 1
    return f"{prefijo}{str(numero).zfill(digitos)}"

def generar_ids_lote(n, prefijo, digitos=4):
    """
    Genera una lista de n IDs consecutivos con prefijo,
    útil para insertar varios registros en lote.
    Ejemplo: generar_ids_lote(4,'P') → ['P0001','P0002','P0003']
    """
    return [f"{prefijo}{str(i).zfill(digitos)}" for i in range(1, n + 1)]

print("✅ Funciones de ID personalizadas definidas")

✅ Funciones de ID personalizadas definidas


In [7]:
# Solicitar cantidad de proveedores
cantidad = int(input("¿Cuántos proveedores deseas insertar? "))

# Datos base para generar proveedores ficticios
nombres_base = [
    "MediSupply", "DermaInsumos", "BioCosméticos", "SaludEstética",
    "ProveeMed", "FarmaDistrib", "CosméticaPro", "InsumosSalud",
    "MediCare", "BioMed", "EstéticaPlus", "SaludTotal",
    "DermaCare", "MediTech", "BioSalud", "FarmaPro"
]

sufijos = ["Colombia", "Ltda", "SA", "SAS", "Express", "Plus", "Corp", "Group"]

dominios = ["medisupply.co", "dermainsumos.co", "biocos.co", "saludest.co",
            "provmed.co", "farma.co", "cosmetica.co", "insumos.co"]

prefijos_correo = ["ventas", "info", "contacto", "soporte", "pedidos", "compras"]

def generar_telefono():
    operadoras = ["300", "310", "315", "320", "321", "311", "312", "313"]
    operadora = random.choice(operadoras)
    numero = random.randint(1000000, 9999999)
    return f"+57 {operadora} {str(numero)[:3]} {str(numero)[3:]}"

def generar_proveedor(i):
    nombre = f"{random.choice(nombres_base)} {random.choice(sufijos)}"
    telefono = generar_telefono()
    dominio = random.choice(dominios)
    correo = f"{random.choice(prefijos_correo)}{i}@{dominio}"
    return (nombre, telefono, correo)

# Generar lista de proveedores
proveedores = [generar_proveedor(i) for i in range(1, cantidad + 1)]

# Insertar en la base de datos
cursor.executemany("""
    INSERT INTO proveedor (Nombre, Telefono, Correo_electronico)
    VALUES (%s, %s, %s)
""", proveedores)
conn.commit()

cursor.execute("SELECT ID_proveedor FROM proveedor")
ids_proveedor = [row[0] for row in cursor.fetchall()]
print(f"✅ Proveedores insertados: {len(ids_proveedor)}")

✅ Proveedores insertados: 10


In [8]:
# ── 5. POBLAR: cliente (sin cambio de PK) ───────────────────

condiciones = ["Diabetes tipo 2", "Hipertensión", "Alergia al látex",
               "Rosácea", "Psoriasis", None, None, None]

clientes_data = []
nombres_usados   = set()
telefonos_usados = set()

for i in range(1000):
    nombre = fake.name()
    while nombre in nombres_usados:
        nombre = fake.name()
    nombres_usados.add(nombre)

    telefono = f"+57 3{random.randint(10,25)} {random.randint(100,999)} {random.randint(1000,9999)}"
    while telefono in telefonos_usados:
        telefono = f"+57 3{random.randint(10,25)} {random.randint(100,999)} {random.randint(1000,9999)}"
    telefonos_usados.add(telefono)

    clientes_data.append((
        nombre, telefono, fake.email(),
        fake.address().replace('\n', ', ')[:50],
        random.choice(condiciones)
    ))

cursor.executemany("""
    INSERT INTO cliente (Nombre, Telefono, Correo_electronico, Direccion, Condicion_medica)
    VALUES (%s, %s, %s, %s, %s)
""", clientes_data)
conn.commit()

cursor.execute("SELECT ID_cliente FROM cliente")
ids_cliente = [row[0] for row in cursor.fetchall()]
print(f"✅ Clientes insertados: {len(ids_cliente)}")

✅ Clientes insertados: 1000


In [9]:
# ── 6. POBLAR: especialista (sin cambio de PK) ─────────────────
especialistas = [
    ("Dra. Laura Mendoza",    "+57 311 601 0001", "l.mendoza@clinica.co",    "Dermatología Clínica"),
    ("Lic. Carlos Ruiz",      "+57 312 602 0002", "c.ruiz@clinica.co",       "Estética Facial Avanzada"),
    ("Dra. Sofía Herrera",    "+57 313 603 0003", "s.herrera@clinica.co",    "Medicina Estética"),
    ("Lic. Andrés Torres",    "+57 314 604 0004", "a.torres@clinica.co",     "Masoterapia Certificada"),
    ("Dra. Valeria Gómez",    "+57 315 605 0005", "v.gomez@clinica.co",      "Tecnología Láser"),
    ("Lic. Camila Vargas",    "+57 316 606 0006", "c.vargas@clinica.co",     "Nutrición Estética"),
    ("Dr. Felipe Morales",    "+57 317 607 0007", "f.morales@clinica.co",    "Medicina Regenerativa"),
    ("Dra. Natalia Ospina",   "+57 318 608 0008", "n.ospina@clinica.co",     "Depilación Láser"),
    ("Lic. Juliana Castillo", "+57 319 609 0009", "j.castillo@clinica.co",   "Cosmetología Avanzada"),
    ("Dr. Sebastián Ríos",    "+57 320 610 0010", "s.rios@clinica.co",       "Cirugía Estética Menor"),
    ("Dra. Marcela Pinto",    "+57 321 611 0011", "m.pinto@clinica.co",      "Dermatología Pediátrica"),
    ("Lic. Diego Salazar",    "+57 322 612 0012", "d.salazar@clinica.co",    "Aromaterapia Clínica"),
    ("Dra. Isabela Ramos",    "+57 323 613 0013", "i.ramos@clinica.co",      "Mesoterapia"),
    ("Lic. Paola Jiménez",    "+57 324 614 0014", "p.jimenez@clinica.co",    "Micropigmentación"),
    ("Dr. Tomás Guerrero",    "+57 325 615 0015", "t.guerrero@clinica.co",   "Fotorrejuvenecimiento"),
]

cursor.executemany("""
    INSERT INTO especialista (Nombre, Telefono, Correo_electronico, Habilidad_certificado)
    VALUES (%s, %s, %s, %s)
""", especialistas)
conn.commit()

cursor.execute("SELECT ID_especialista FROM especialista")
ids_especialista = [row[0] for row in cursor.fetchall()]
print(f"✅ Especialistas insertados: {len(ids_especialista)}")

✅ Especialistas insertados: 15


In [10]:
# ── 7. POBLAR: producto → PK formato "P0001" ───────────────
productos_raw = [
    ("Sérum Vitamina C",          "Sérum antioxidante con 20% vitamina C pura",                    85000),
    ("Ácido Hialurónico 1ml",     "Relleno dérmico de ácido hialurónico estéril",                 220000),
    ("Mascarilla Hidratante",     "Mascarilla de alginato con colágeno marino",                    35000),
    ("Filtro Solar SPF 50",       "Protector solar de amplio espectro, no comedogénico",           42000),
    ("Crema Despigmentante",      "Crema con kojic acid y niacinamida al 5%",                      67000),
    ("Gel Conductor Ultrasonido", "Gel neutro para tratamientos con ultrasonido",                  18000),
    ("Peeling Enzimático",        "Exfoliante enzimático con papaína y bromelina",                 55000),
    ("Ampolla Retinol 0.5%",      "Tratamiento nocturno con retinol microencapsulado",             98000),
    ("Tónico Ácido Glicólico",    "Tónico exfoliante con AHA al 7% para piel opaca",              48000),
    ("Contorno de Ojos",          "Gel-crema con cafeína y péptidos para bolsas y ojeras",         72000),
    ("Aceite de Rosa Mosqueta",   "Aceite vegetal prensado en frío, cicatrizante y regenerador",   39000),
    ("Mascarilla de Carbón",      "Mascarilla purificante con carbón activado y árbol de té",      31000),
    ("Crema Corporal Reafirmante","Emulsión con colágeno hidrolizado y manteca de karité",         54000),
    ("Ampolla Vitamina B12",      "Ampolla inyectable de soporte metabólico dérmico",             115000),
    ("Espuma Limpiadora",         "Limpiador espumoso con prebióticos para piel sensible",         29000),
    ("Suero Antimanchas",         "Sérum con ácido tranexámico al 3% y extracto de regaliz",       88000),
    ("Gel Hidratante Aloe Vera",  "Gel calmante 99% aloe vera orgánico certificado",               22000),
    ("Crema FPS 30 con Color",    "BB cream con protección solar y cobertura ligera",              47000),
    ("Ampolla Placenta Vegetal",  "Ampolla revitalizante con proteínas vegetales hidrolizadas",    130000),
    ("Kit Microagujas 0.5mm",     "Dermaroller estéril de uso profesional con 540 agujas",         95000),
]

# Generamos los IDs: P0001, P0002 … P0020
ids_producto = generar_ids_lote(len(productos_raw), prefijo="P")

# Empaquetamos (ID, Nombre, Descripcion, Precio, ID_proveedor)
productos_data = [
    (ids_producto[i], n, d, p, random.choice(ids_proveedor))
    for i, (n, d, p) in enumerate(productos_raw)
]

cursor.executemany("""
    INSERT INTO producto (ID_producto, Nombre, Descripcion, Precio, ID_proveedor)
    VALUES (%s, %s, %s, %s, %s)
""", productos_data)
conn.commit()
print(f"✅ Productos insertados con IDs: {ids_producto}")

✅ Productos insertados con IDs: ['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008', 'P0009', 'P0010', 'P0011', 'P0012', 'P0013', 'P0014', 'P0015', 'P0016', 'P0017', 'P0018', 'P0019', 'P0020']


In [11]:
# ── 8. POBLAR: procedimiento_servicio → PK formato "PS0001" ─
procedimientos_raw = [
    ("Limpieza Facial Profunda",    "Limpieza con vapor, extracción y mascarilla calmante",          1.50, 120000),
    ("Peeling Químico Medio",       "Aplicación ácido glicólico 30% con neutralización",             1.00, 180000),
    ("Radiofrecuencia Facial",      "Tratamiento tensor con radiofrecuencia monopolar",               1.00, 250000),
    ("Microagujamiento Dérmico",    "Inducción de colágeno con microagujas de 1.5mm",                 1.50, 320000),
    ("Hidratación Profunda",        "Protocolo hidratación con ultrasonido e iontoforesis",           0.75, 150000),
    ("Depilación Láser Facial",     "Depilación láser Alexandrita zona facial completa",              0.50, 200000),
    ("Masaje Facial Kobido",        "Masaje japonés lifting natural con técnica Kobido",              1.00, 160000),
    ("Tratamiento Anti-Acné",       "Protocolo con luz LED azul, ácido salicílico y ozono",           1.25, 210000),
    ("Mesoterapia Facial",          "Microinyecciones de cóctel vitamínico revitalizante",            1.00, 280000),
    ("Carboxiterapia",              "Infiltración de CO₂ medicinal para ojeras y flacidez",           0.75, 230000),
    ("Fotorrejuvenecimiento IPL",   "Luz pulsada intensa para manchas, rojeces y poros",              1.00, 300000),
    ("Plasma Rico en Plaquetas",    "Aplicación de PRP autólogo para regeneración celular",           1.50, 420000),
    ("Drenaje Linfático Facial",    "Técnica manual Vodder para reducir retención e inflamación",     1.00, 140000),
    ("Peeling de Retinol",          "Peeling nocturno con retinol 1% y ácido mandélico",              0.75, 195000),
    ("Ultracavitación Facial",      "Cavitación focalizada para redefinir el óvalo facial",           1.00, 270000),
]

# Generamos los IDs: PS0001, PS0002 … PS0015
ids_procedimiento = generar_ids_lote(len(procedimientos_raw), prefijo="PS")

procedimientos_data = [
    (ids_procedimiento[i], n, d, dur, p)
    for i, (n, d, dur, p) in enumerate(procedimientos_raw)
]

cursor.executemany("""
    INSERT INTO procedimiento_servicio (ID_procedimiento, Nombre, Descripcion, Duracion_promedio, Precio)
    VALUES (%s, %s, %s, %s, %s)
""", procedimientos_data)
conn.commit()
print(f"✅ Procedimientos insertados con IDs: {ids_procedimiento}")

✅ Procedimientos insertados con IDs: ['PS0001', 'PS0002', 'PS0003', 'PS0004', 'PS0005', 'PS0006', 'PS0007', 'PS0008', 'PS0009', 'PS0010', 'PS0011', 'PS0012', 'PS0013', 'PS0014', 'PS0015']


In [12]:
# ── 9. POBLAR: inventario  →  PK formato "I001" ──────────────
# ⚠️  ALTER TABLE inventario MODIFY ID_inventario VARCHAR(10);
#
# Precargamos duraciones desde el dict ya construido en memoria.

duraciones = {
    ids_procedimiento[i]: dur
    for i, (_, _, dur, _) in enumerate(procedimientos_raw)
}

inventario_raw = []
fecha_base = date(2024, 1, 1)

for id_prod in ids_producto:
    for _ in range(500):
        inventario_raw.append((
            random.randint(10, 100),
            random.randint(1,  30),
            fecha_base + timedelta(days=random.randint(0, 365)),
            id_prod                   # FK → producto (VARCHAR "P00x")
        ))

# Generamos IDs: I001, I002 … I0XX
ids_inventario = generar_ids_lote(len(inventario_raw), prefijo="I")

inventario_data = [
    (ids_inventario[i], ent, sal, fec, prod)
    for i, (ent, sal, fec, prod) in enumerate(inventario_raw)
]

cursor.executemany("""
    INSERT INTO inventario
        (ID_inventario, Cantidad_entrante, Cantidad_saliente, Fecha_movimiento, ID_Producto)
    VALUES (%s, %s, %s, %s, %s)
""", inventario_data)
conn.commit()
print(f"✅ Inventario insertado — {len(inventario_data)} registros, IDs I001…{ids_inventario[-1]}")

✅ Inventario insertado — 10000 registros, IDs I001…I10000


In [ ]:
# ── 10. POBLAR: cita  →  PK formato "C0001" ───────────────────
# ⚠️  ALTER TABLE cita MODIFY ID_cita VARCHAR(10);
#     ALTER TABLE factura MODIFY ID_cita VARCHAR(10);
#
# Hora_final = Hora_inicio + duración del procedimiento elegido.

citas_raw = []
fecha_cita_base = date(2024, 3, 1)

for _ in range(1000):
    id_cli  = random.choice(ids_cliente)
    id_proc = random.choice(ids_procedimiento)   # "PS000x"
    id_esp  = random.choice(ids_especialista)
    fecha   = fecha_cita_base + timedelta(days=random.randint(0, 200))

    hora_inicio = datetime.strptime(
        f"{random.randint(8,16):02d}:{random.choice(['00','30'])}:00", "%H:%M:%S"
    )
    hora_final = hora_inicio + timedelta(hours=duraciones[id_proc])

    citas_raw.append((
        fecha, hora_inicio.time(), hora_final.time(),
        random.choice([True, False]),
        id_cli, id_proc, id_esp
    ))

# Generamos IDs: C001, C002 … C015
ids_cita = generar_ids_lote(len(citas_raw), prefijo="C")

citas_data = [
    (ids_cita[i], fec, hi, hf, est, cli, proc, esp)
    for i, (fec, hi, hf, est, cli, proc, esp) in enumerate(citas_raw)
]

cursor.executemany("""
    INSERT INTO cita
        (ID_cita, Fecha, Hora_inicio, Hora_final, Estado,
         ID_cliente, ID_procedimiento, ID_especialista)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
""", citas_data)
conn.commit()
print(f"✅ Citas insertadas con IDs: {ids_cita}")

✅ Citas insertadas con IDs: ['C0001', 'C0002', 'C0003', 'C0004', 'C0005', 'C0006', 'C0007', 'C0008', 'C0009', 'C0010', 'C0011', 'C0012', 'C0013', 'C0014', 'C0015', 'C0016', 'C0017', 'C0018', 'C0019', 'C0020', 'C0021', 'C0022', 'C0023', 'C0024', 'C0025', 'C0026', 'C0027', 'C0028', 'C0029', 'C0030', 'C0031', 'C0032', 'C0033', 'C0034', 'C0035', 'C0036', 'C0037', 'C0038', 'C0039', 'C0040', 'C0041', 'C0042', 'C0043', 'C0044', 'C0045', 'C0046', 'C0047', 'C0048', 'C0049', 'C0050', 'C0051', 'C0052', 'C0053', 'C0054', 'C0055', 'C0056', 'C0057', 'C0058', 'C0059', 'C0060', 'C0061', 'C0062', 'C0063', 'C0064', 'C0065', 'C0066', 'C0067', 'C0068', 'C0069', 'C0070', 'C0071', 'C0072', 'C0073', 'C0074', 'C0075', 'C0076', 'C0077', 'C0078', 'C0079', 'C0080', 'C0081', 'C0082', 'C0083', 'C0084', 'C0085', 'C0086', 'C0087', 'C0088', 'C0089', 'C0090', 'C0091', 'C0092', 'C0093', 'C0094', 'C0095', 'C0096', 'C0097', 'C0098', 'C0099', 'C0100', 'C0101', 'C0102', 'C0103', 'C0104', 'C0105', 'C0106', 'C0107', 'C0108',

In [14]:
# ── 11. POBLAR: factura ───────────────────────────────────────
# FK → ID_cita (VARCHAR "C00x") + ID_producto (VARCHAR "P00x")
# La PK de factura puede seguir siendo AUTO_INCREMENT.

factura_data = []
for id_cita in ids_cita:
    for _ in range(random.randint(1, 3)):
        factura_data.append((
            id_cita,
            random.choice(ids_producto),
            random.randint(1, 5)
        ))

cursor.executemany("""
    INSERT INTO factura (ID_cita, ID_producto, Cantidad)
    VALUES (%s, %s, %s)
""", factura_data)
conn.commit()
print(f"✅ Líneas de factura insertadas: {len(factura_data)}")

✅ Líneas de factura insertadas: 2038


In [15]:
# ── 12. POBLAR: producto_procedimiento ───────────────────────
# PK compuesta: FK "PS000x" + FK "P000x" → ambas VARCHAR.
# El set evita duplicar la PK compuesta.

pp_set = set()
for id_proc in ids_procedimiento:
    for id_prod in random.sample(ids_producto, k=random.randint(1, 3)):
        pp_set.add((id_proc, id_prod, random.randint(1, 4)))

cursor.executemany("""
    INSERT INTO producto_procedimiento (ID_procedimiento, ID_producto, cantidad_producto)
    VALUES (%s, %s, %s)
""", list(pp_set))
conn.commit()
print(f"✅ Relaciones producto-procedimiento: {len(pp_set)}")

✅ Relaciones producto-procedimiento: 27


In [16]:
# ── 13. VERIFICACIÓN FINAL ───────────────────────────────────

tablas = [
    "proveedor", "cliente", "especialista", "producto",
    "procedimiento_servicio", "inventario", "cita",
    "factura", "producto_procedimiento"
]

print("\n📋 RESUMEN FINAL DE REGISTROS:")
print("─" * 40)
for tabla in tablas:
    cursor.execute(f"SELECT COUNT(*) FROM {tabla}")
    print(f"  {tabla:<30} → {cursor.fetchone()[0]:>3} registros")

# Muestra de IDs generados por prefijo
print("\n🔑 MUESTRA DE IDs PERSONALIZADOS:")
for tabla, col, prefijo in [
    ("producto",               "ID_producto",      "P"),
    ("procedimiento_servicio", "ID_procedimiento",  "PS"),
    ("inventario",             "ID_inventario",     "I"),
    ("cita",                   "ID_cita",           "C"),
]:
    cursor.execute(f"SELECT {col} FROM {tabla} LIMIT 3")
    muestra = [r[0] for r in cursor.fetchall()]
    print(f"  {prefijo}xxx  →  {muestra}")

cursor.close()
conn.close()
print("\n🔒 Conexión cerrada correctamente")


📋 RESUMEN FINAL DE REGISTROS:
────────────────────────────────────────
  proveedor                      →  10 registros
  cliente                        → 1000 registros
  especialista                   →  15 registros
  producto                       →  20 registros
  procedimiento_servicio         →  15 registros
  inventario                     → 10000 registros
  cita                           → 1000 registros
  factura                        → 2038 registros
  producto_procedimiento         →  27 registros

🔑 MUESTRA DE IDs PERSONALIZADOS:
  Pxxx  →  ['P0015', 'P0012', 'P0019']
  PSxxx  →  ['PS0010', 'PS0006', 'PS0013']
  Ixxx  →  ['I0001', 'I0002', 'I0003']
  Cxxx  →  ['C0862', 'C0251', 'C0425']

🔒 Conexión cerrada correctamente
